# 結晶検出（YOLO） - 学習・評価（Jupyter版）
`data/images/` + `data/labels/` + `data/data.yaml`（`colab/yolo_dataset_pipeline.ipynb`でエクスポートしたもの）
を使って、事前学習済みYOLOをファインチューニングする。`yolo_project/`フォルダを
カレントディレクトリとしてこのnotebookを実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = "data/data.yaml"
MODEL = "yolov8n.pt"   # 事前学習済み・軽量モデル。精度が足りなければ yolov8s.pt / yolov8m.pt へ
EPOCHS = 100
IMGSZ = 640
BATCH = 16
NAME = "crystal_yolo"

assert Path(DATA_YAML).exists(), f"{DATA_YAML} が見つかりません。data/ にデータセットを展開してください。"
print("準備完了")

## 学習

In [ ]:
model = YOLO(MODEL)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=NAME,
)
best_weights = model.trainer.save_dir / "weights" / "best.pt"
print(f"\n保存先: {best_weights}")

## 評価（mAP・Precision・Recall）

In [ ]:
eval_model = YOLO(str(best_weights))
metrics = eval_model.val(data=DATA_YAML, imgsz=IMGSZ)

print(f"mAP50    = {metrics.box.map50:.4f}")
print(f"mAP50-95 = {metrics.box.map:.4f}")
print(f"Precision = {metrics.box.mp:.4f}")
print(f"Recall    = {metrics.box.mr:.4f}")

## 予測結果を目視確認

In [ ]:
import random
import matplotlib.pyplot as plt

val_img_dir = Path(DATA_YAML).parent / "images" / "val"
all_val_images = list(val_img_dir.glob("*.png"))
sample_paths = random.sample(all_val_images, min(4, len(all_val_images)))

preds = eval_model.predict(sample_paths, conf=0.25, imgsz=IMGSZ)

fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
if len(preds) == 1:
    axes = [axes]
for ax, r in zip(axes, preds):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()
plt.show()